# 10.6 - Remote MCP on Cloud Run

**Module 10 - Tool Use & MCP** | Netsetos GenAI Engineering

This notebook works through Remote MCP on Cloud Run hands-on: Serve the server over HTTP; Why stateless routing scales; A lean Dockerfile for fast cold starts; Model the cold-start cost of image size; Deploy to Cloud Run from source; Model autoscaling 0 -> N.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the one non-standard package this notebook can touch. Everything else here is either a file being written to disk (`%%writefile`) or a pure-Python simulation, so the runtime footprint is deliberately tiny.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


**Explanation**

The single pip line pulls in `python-dotenv`, which the next cell can use to read keys from a `.env` file. It is commented out because on a normal machine you already have it; uncomment it on a fresh Colab or clean environment.

**How the code works - step by step**
- **`!pip install python-dotenv -q`** - installs the dotenv loader quietly; the `# noqa` marks the shell-magic line so linters skip it.

**In one line:** one optional install, no imports yet.

**What you'll see:** (no output - environment prep)

## Setup - load provider keys

Read any LLM provider keys from the environment so the client-connection example (Step 9) has them if you extend it into a real call. Nothing in this notebook actually calls an API, so blank keys are fine to run everything.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

This cell only stages environment variables; it does not authenticate anything. `setdefault` writes an empty string only if the variable is unset, so a real key already exported in your shell is left untouched. Read it as a placeholder that keeps secrets out of the code.

**How the code works - step by step**
- **`import os`** - gives access to the process environment.
- **`os.environ.setdefault("OPENAI_API_KEY", "")`** and the ANTHROPIC / GOOGLE lines - seed each key with an empty default, never overwriting a real one you already set.

**In one line:** stage provider keys from the environment without hardcoding any.

**What you'll see:** (no output - environment prep)

## 1 - Serve the server over HTTP

This is the whole "go remote" move: take the Lesson 10.5 server and change the one line that decides *how* it is reachable. `%%writefile` saves it to `01_http_server.py` on disk - the exact file the Dockerfile will run inside a container.

> **Needs `pip install fastmcp` to actually run** - as written it only writes the file.

In [ ]:
%%writefile 01_http_server.py
# GO REMOTE - the SAME 10.5 server, now served over Streamable HTTP for Cloud Run.
# pip install fastmcp
import os
from fastmcp import FastMCP

# stateless_http=True: each request is self-contained (no server-side session),
# so ANY Cloud Run instance can serve ANY request - horizontal scaling just works.
mcp = FastMCP("Netsetos", stateless_http=True)

@mcp.tool()
def add_gst(amount: float) -> float:
    "Add 18% GST to a rupee amount."
    return round(amount * 1.18, 2)

if __name__ == "__main__":
    # transport="http" IS Streamable HTTP (older alias: "streamable-http").
    # Cloud Run REQUIRES host="0.0.0.0" and the injected $PORT.
    mcp.run(transport="http", host="0.0.0.0", port=int(os.environ.get("PORT", 8080)))

# Output: (illustrative - needs `pip install fastmcp`) the server now serves MCP over
#   http://0.0.0.0:$PORT/mcp - the SAME tools as 10.5, reachable remotely. For an ASGI app
#   (to add middleware), use `app = mcp.http_app()` and run it with uvicorn instead.

**Explanation**

The tool code from 10.5 (`add_gst`) is byte-identical; only the transport changed. Swapping `stdio` for `transport="http"` turns the server into a Streamable-HTTP service, and `stateless_http=True` removes the per-session state so any Cloud Run instance can answer any request. This is a config change, not a rewrite.

**How the code works - step by step**
- **`FastMCP("Netsetos", stateless_http=True)`** - creates the server with no server-side session, the flag that unlocks horizontal scaling.
- **`@mcp.tool() def add_gst`** - the same tool as 10.5, unchanged; the function and its schema are identical.
- **`mcp.run(transport="http", host="0.0.0.0", port=...)`** - serves Streamable HTTP; `0.0.0.0` makes it reachable outside the container and `$PORT` uses the port Cloud Run injects (falls back to 8080 locally).
- **trailing comment** - notes that for middleware you'd use `app = mcp.http_app()` under uvicorn instead.

**In one line:** same tools, one transport argument, bound to `0.0.0.0:$PORT`.

**What you'll see:** Writes the file: `Writing 01_http_server.py`. No server actually starts (and it would need fastmcp installed); the code comment describes the illustrative endpoint at `http://0.0.0.0:$PORT/mcp`.

## 2 - Why stateless routing scales

A tiny simulation that shows, in numbers, what `stateless_http=True` buys you. It routes six requests (from three sessions) across three instances two ways and compares the load spread. This is a measurement toy, not a real server.

In [ ]:
# STATELESS vs STATEFUL routing - why stateless_http=True unlocks Cloud Run scaling.
def route(requests, instances, stateful):
    placed, sessions = [], {}
    for i, sid in enumerate(requests):
        if stateful:
            inst = sessions.setdefault(sid, len(sessions) % instances)  # pin session -> one instance
        else:
            inst = i % instances                                        # any instance serves any request
        placed.append(inst)
    return placed

reqs = ["s1", "s2", "s1", "s3", "s2", "s1"]        # 3 sessions, 6 requests, 3 instances
st = route(reqs, 3, stateful=True)
sl = route(reqs, 3, stateful=False)
print(f"  stateful  -> instances used: {sorted(set(st))}  load per instance: {[st.count(i) for i in range(3)]}")
print(f"  stateless -> instances used: {sorted(set(sl))}  load per instance: {[sl.count(i) for i in range(3)]}")
print("  stateful pins each session to one instance (needs affinity, uneven load);")
print("  stateless spreads every request across all instances -> Cloud Run scales freely.")

# Output:
#   stateful  -> instances used: [0, 1, 2]  load per instance: [3, 2, 1]
#   stateless -> instances used: [0, 1, 2]  load per instance: [2, 2, 2]
#   stateful pins each session to one instance (needs affinity, uneven load);
#   stateless spreads every request across all instances -> Cloud Run scales freely.

**Explanation**

The `route` function is a stand-in for a load balancer: in stateful mode it pins each session to a fixed instance; in stateless mode it hands each request to the next instance in turn. Running both over the same request stream makes the fairness difference concrete - stateful clumps load, stateless spreads it evenly.

**How the code works - step by step**
- **`route(requests, instances, stateful)`** - loops over the requests and assigns each to an instance.
- **stateful branch** - `sessions.setdefault(sid, ...)` pins a session id to one instance for its whole life (needs affinity).
- **stateless branch** - `i % instances` round-robins every request regardless of session.
- **the two `print` lines** - report which instances were used and the per-instance load for each mode.

**In one line:** stateful pins sessions (lumpy load); stateless round-robins (even load).

**What you'll see:** Prints `stateful -> ... load per instance: [3, 2, 1]` versus `stateless -> ... load per instance: [2, 2, 2]`, plus two lines explaining that even spread is what lets Cloud Run scale freely.

## 3 - A lean Dockerfile for fast cold starts

Cloud Run runs containers, and on Cloud Run cold-start time is billable and user-facing, so the image must be small. This `%%writefile` saves Google's reference pattern: a slim Python base plus the `uv` installer, with dependencies layered before source so the build cache survives code edits.

In [ ]:
%%writefile Dockerfile
# Dockerfile - lean base + uv for a fast Cloud Run cold start (Google's reference pattern).
FROM python:3.13-slim
# copy the uv binary (far faster than pip); UV_COMPILE_BYTECODE precompiles on install
COPY --from=ghcr.io/astral-sh/uv:latest /uv /uvx /bin/
ENV UV_COMPILE_BYTECODE=1
WORKDIR /app
# install deps FIRST so the layer cache survives source-only changes
COPY pyproject.toml uv.lock ./
RUN uv sync --frozen --no-dev
COPY . .
# run as non-root (Cloud Run does not require it, but it is the baseline)
RUN useradd -m appuser && chown -R appuser /app
USER appuser
# Cloud Run injects $PORT; the server binds 0.0.0.0:$PORT (see 01_http_server.py)
CMD ["uv", "run", "01_http_server.py"]

# Output: (illustrative) builds to a ~270 MB image (vs ~1 GB for python:3.13 full),
#   so Cloud Run pulls and cold-starts it several times faster.

**Explanation**

This is a build recipe, not runnable Python - it defines how the server gets packaged. The two ideas that matter are the small base (`python:3.13-slim` instead of the ~1 GB full image) and the layer ordering (install deps first, copy source last) so that editing your code doesn't re-run the dependency install.

**How the code works - step by step**
- **`FROM python:3.13-slim`** - the small base image.
- **`COPY --from=ghcr.io/astral-sh/uv ...`** and **`ENV UV_COMPILE_BYTECODE=1`** - bring in the fast `uv` installer and precompile bytecode for a quicker start.
- **`COPY pyproject.toml uv.lock` then `RUN uv sync --frozen --no-dev`** - install deps as a cached layer *before* the source is copied.
- **`COPY . .`** - add the application code (invalidates only this later layer).
- **`useradd appuser` + `USER appuser`** - run as non-root, the security baseline.
- **`CMD ["uv", "run", "01_http_server.py"]`** - launch the Step 1 server, which binds `0.0.0.0:$PORT`.

**In one line:** slim base + uv + deps-before-source = a ~270 MB image that cold-starts fast.

**What you'll see:** Writes the file: `Writing Dockerfile`. No build runs here; the comment notes the target of a ~270 MB image versus ~1 GB for the full base.

## 4 - Model the cold-start cost of image size

A back-of-envelope model that turns image size into a cold-start pull time, comparing the full base + pip against slim + uv. It quantifies the tradeoff the Dockerfile is chasing - it is an illustrative model, not a benchmark.

In [ ]:
# IMAGE SIZE + COLD-START model - full vs slim+uv (an illustrative model, not a benchmark).
def build(base_mb, deps_mb, tool):        # tool = "pip" or "uv"
    image_mb = base_mb + deps_mb
    pull_s = image_mb / 200.0             # ~200 MB/s registry pull (model)
    return image_mb, round(pull_s, 2), ("uv (fast)" if tool == "uv" else "pip (slow)")

full, full_pull, _ = build(1000, 120, "pip")    # python:3.13 full
slim, slim_pull, n = build(150, 120, "uv")      # python:3.13-slim + uv
print(f"  full  image ~{full} MB -> pull ~{full_pull}s   (python:3.13 + pip)")
print(f"  slim  image ~{slim} MB -> pull ~{slim_pull}s   (python:3.13-slim + {n})")
print(f"  slim+uv is ~{round(full/slim,1)}x smaller -> a materially faster cold start.")
print("  layer cache: change only source -> deps layer is reused, rebuild is near-instant.")

# Output:
#   full  image ~1120 MB -> pull ~5.6s   (python:3.13 + pip)
#   slim  image ~270 MB -> pull ~1.35s   (python:3.13-slim + uv (fast))
#   slim+uv is ~4.1x smaller -> a materially faster cold start.
#   layer cache: change only source -> deps layer is reused, rebuild is near-instant.

**Explanation**

The `build` function estimates image size and a registry pull time from a fixed pull-rate assumption; running it twice (full vs slim) shows the multiplier. The point is the *shape*: a ~4x smaller image means a materially faster cold start, and the layer-cache note reminds you that source-only edits skip the dependency rebuild entirely.

**How the code works - step by step**
- **`build(base_mb, deps_mb, tool)`** - adds base + deps to get image size, divides by an assumed ~200 MB/s to get pull seconds, and labels the installer.
- **`build(1000, 120, "pip")`** - the full `python:3.13` + pip case.
- **`build(150, 120, "uv")`** - the slim + uv case.
- **the `print` block** - shows both sizes and pull times, the size ratio, and the cache reminder.

**In one line:** smaller image -> faster pull -> faster cold start, and cached deps make rebuilds near-instant.

**What you'll see:** Prints `full image ~1120 MB -> pull ~5.6s` and `slim image ~270 MB -> pull ~1.35s`, then `slim+uv is ~4.1x smaller`, plus the layer-cache note.

## 5 - Deploy to Cloud Run from source

The actual ship command, private by default. `%%bash` marks it as shell (run it in a real terminal with `gcloud` configured, not in Colab). Cloud Run builds the container from your Dockerfile, gives it HTTPS and a `*.run.app` URL, and auto-scales from zero.

In [ ]:
%%bash
# (deployment/terminal commands - run in a shell, not Colab, unless configured)
# DEPLOY TO CLOUD RUN - from source, PRIVATE by default (Google's reference path).
# Cloud Run builds the container, gives it HTTPS + a *.run.app URL, and scales 0->N.
gcloud run deploy netsetos-mcp \
  --source . \
  --region us-central1 \
  --no-allow-unauthenticated \
  --min-instances 0 \
  --max-instances 10 \
  --cpu-boost \
  --timeout=300
# --timeout=300 (default) bounds any one request or SSE stream; the hard max is 3600s (60 min).
# --no-allow-unauthenticated = PRIVATE (not open to the internet).
# --min-instances 0 = scale to zero when idle (~$0). --cpu-boost = faster cold start.

# the MCP endpoint is <service-url>/mcp
gcloud run services describe netsetos-mcp --region us-central1 --format "value(status.url)"

# Output: (illustrative)
#   Building with Cloud Build... Deploying... Done.
#   Service [netsetos-mcp] revision [netsetos-mcp-00001] deployed, serving 100% traffic.
#   Service URL: https://netsetos-mcp-abc123-uc.a.run.app   (MCP endpoint: .../mcp)

**Explanation**

This cell is a deployment script, not Python - running the notebook cell without a configured `gcloud` won't deploy anything. The critical flag is `--no-allow-unauthenticated`, which ships the service *private*: the URL exists but rejects anonymous calls until Step 7 grants access. `--source .` skips the manual Artifact Registry push by letting Cloud Run build for you.

**How the code works - step by step**
- **`gcloud run deploy netsetos-mcp --source .`** - build-and-deploy straight from the current directory using the Dockerfile.
- **`--no-allow-unauthenticated`** - deploy PRIVATE (locked until you explicitly grant access).
- **`--min-instances 0` / `--max-instances 10`** - scale to zero when idle, cap the fan-out at 10.
- **`--cpu-boost` / `--timeout=300`** - extra CPU during startup; bound any single request/stream at 300s (hard max 3600s).
- **`gcloud run services describe ... --format "value(status.url)"`** - print the deployed HTTPS URL; the MCP endpoint is that URL + `/mcp`.

**In one line:** one command builds, privately deploys, and auto-scales the server, then prints its URL.

**What you'll see:** Illustrative build/deploy log ending in `Service URL: https://netsetos-mcp-abc123-uc.a.run.app` (MCP endpoint `.../mcp`). Nothing deploys unless you have gcloud set up.

## 6 - Model autoscaling 0 -> N

A simulation of how Cloud Run turns request load into an instance count, using Little's law and concurrency. It shows the scale-to-zero floor and the max-instances ceiling in one sweep - a model, not live metrics.

In [ ]:
# AUTOSCALE 0 -> N (scale to zero) - how Cloud Run turns traffic into instances.
def instances_needed(rps, latency_s, concurrency, min_inst=0, max_inst=10):
    in_flight = rps * latency_s                    # Little's law: concurrent requests
    need = -(-int(in_flight) // concurrency)       # ceil division
    return max(min_inst, min(need, max_inst))

for rps in [0, 5, 80, 400, 2000]:
    n = instances_needed(rps, latency_s=0.4, concurrency=80)
    tag = "scale to ZERO (idle, ~$0)" if n == 0 else ("capped at max" if n == 10 else "auto-scaled")
    print(f"  {rps:>4} req/s -> {n:>2} instance(s)   {tag}")
print("  concurrency=80 packs many I/O-bound tool calls per instance; idle -> 0 instances.")

# Output:
#      0 req/s ->  0 instance(s)   scale to ZERO (idle, ~$0)
#      5 req/s ->  1 instance(s)   auto-scaled
#     80 req/s ->  1 instance(s)   auto-scaled
#    400 req/s ->  2 instance(s)   auto-scaled
#   2000 req/s -> 10 instance(s)   capped at max
#   concurrency=80 packs many I/O-bound tool calls per instance; idle -> 0 instances.

**Explanation**

`instances_needed` computes concurrent in-flight requests (`rps x latency`) and divides by per-instance concurrency to get the instance count, clamped between the min and max. Sweeping across a range of request rates makes the whole curve visible: zero at idle, linear in the middle, flat at the cap.

**How the code works - step by step**
- **`in_flight = rps * latency_s`** - Little's law: how many requests are concurrently being served.
- **`need = ceil(in_flight / concurrency)`** - instances required to hold that load (ceil via the `-(-a // b)` trick).
- **`max(min_inst, min(need, max_inst))`** - clamp to the `--min-instances`/`--max-instances` bounds.
- **the loop over `[0, 5, 80, 400, 2000]`** - tags each result as scale-to-zero, auto-scaled, or capped.

**In one line:** requests-per-second -> instance count, floored at zero and capped at max.

**What you'll see:** Prints a table from `0 req/s -> 0 instance(s) scale to ZERO` up to `2000 req/s -> 10 instance(s) capped at max`, plus a note that concurrency=80 packs many I/O-bound calls per instance.

## 7 - Grant access to the private service

The service is private from Step 5; this `%%bash` script shows the service-to-service way to let a specific caller in - Cloud Run IAM, no shared secrets. Run it in a shell with `gcloud`, not in Colab.

In [ ]:
%%bash
# (deployment/terminal commands - run in a shell, not Colab, unless configured)
# SECURE IT - the server is already private (--no-allow-unauthenticated).
# 1) let a specific caller invoke it (service-to-service, no shared secret):
gcloud run services add-iam-policy-binding netsetos-mcp \
  --region us-central1 \
  --member "serviceAccount:caller@PROJECT.iam.gserviceaccount.com" \
  --role roles/run.invoker

# 2) dev shortcut: an authenticated tunnel to localhost (Cloud Run signs every request):
gcloud run services proxy netsetos-mcp --region us-central1
#   -> your local MCP client connects to http://localhost:8080/mcp

# 3) a raw client sends a Google-signed identity token on each request:
#   Authorization: Bearer $(gcloud auth print-identity-token)

# For EXTERNAL end-user clients, front it with OAuth 2.1 + PKCE instead (see prose):
#   the server publishes /.well-known/oauth-protected-resource (RFC 9728);
#   the client sends the resource indicator (RFC 8707) so tokens cannot cross servers.

# Output: (illustrative)
#   Updated IAM policy for service [netsetos-mcp].
#   Proxying to Cloud Run service [netsetos-mcp] on http://localhost:8080

**Explanation**

This is a security recipe, not Python. It grants one caller the `run.invoker` role so Google mints a signed identity token per request, shows the `services proxy` dev shortcut that tunnels an authenticated connection to localhost, and (in comments) points external end-user clients toward OAuth 2.1 + PKCE instead of IAM.

**How the code works - step by step**
- **`add-iam-policy-binding ... --role roles/run.invoker`** - lets one named service account invoke the private service.
- **`gcloud run services proxy netsetos-mcp`** - dev shortcut: an authenticated tunnel so a local client talks to `http://localhost:8080/mcp`.
- **`Authorization: Bearer $(gcloud auth print-identity-token)`** - how a raw client authenticates each request.
- **the OAuth comment block** - external clients publish `/.well-known/oauth-protected-resource` (RFC 9728) and send the resource indicator (RFC 8707) for token isolation.

**In one line:** IAM `run.invoker` for GCP-internal callers; OAuth 2.1 for external users.

**What you'll see:** Illustrative output: `Updated IAM policy for service [netsetos-mcp].` and `Proxying to Cloud Run service ... on http://localhost:8080`. Nothing changes without a real gcloud project.

## 8 - Choose the right auth mechanism

A decision function that maps a caller's situation to the correct auth mode. It encodes the Step 7 rules so you can see, case by case, when to reach for IAM versus OAuth versus public - a pure decision table, no network calls.

In [ ]:
# WHICH AUTH? - a decision function: internal service call vs external end user.
def choose_auth(caller_in_gcp, needs_end_user_identity, is_public_demo):
    if is_public_demo:
        return "public (--allow-unauthenticated)", "ONLY a throwaway demo; anyone with the URL can call it"
    if caller_in_gcp and not needs_end_user_identity:
        return "Cloud Run IAM (roles/run.invoker + identity token)", "service-to-service inside GCP, no shared secrets"
    return "OAuth 2.1 + PKCE (RFC 9728/8707)", "external end-user clients needing per-user identity + token isolation"

cases = [
    ("backend microservice calling the MCP server", True,  False, False),
    ("Claude Desktop user connecting from home",     False, True,  False),
    ("quick hackathon demo, no data",                False, False, True),
]
for label, gcp, user, demo in cases:
    mode, why = choose_auth(gcp, user, demo)
    print(f"  {label}\n    -> {mode}\n       ({why})")

# Output:
#   backend microservice calling the MCP server
#     -> Cloud Run IAM (roles/run.invoker + identity token)
#        (service-to-service inside GCP, no shared secrets)
#   Claude Desktop user connecting from home
#     -> OAuth 2.1 + PKCE (RFC 9728/8707)
#        (external end-user clients needing per-user identity + token isolation)
#   quick hackathon demo, no data
#     -> public (--allow-unauthenticated)
#        (ONLY a throwaway demo; anyone with the URL can call it)

**Explanation**

`choose_auth` is a small rules engine over three booleans: is the caller inside GCP, does it need per-user identity, is it just a throwaway demo. Running it over three realistic cases shows that public is reserved for demos, IAM covers service-to-service, and OAuth handles external end users - the same logic the prose argues, made executable.

**How the code works - step by step**
- **`if is_public_demo`** - returns public (`--allow-unauthenticated`) with a warning that anyone with the URL can call it.
- **`if caller_in_gcp and not needs_end_user_identity`** - returns Cloud Run IAM (`run.invoker` + identity token).
- **else branch** - returns OAuth 2.1 + PKCE (RFC 9728/8707) for external per-user clients.
- **the `cases` loop** - runs a microservice, a Claude Desktop user, and a hackathon demo through the function.

**In one line:** demo -> public, GCP-internal -> IAM, external user -> OAuth.

**What you'll see:** Prints each case with its chosen mode and reason: the microservice -> Cloud Run IAM, the Claude Desktop user -> OAuth 2.1 + PKCE, the hackathon demo -> public (with the warning).

## 9 - Connect remote clients to one URL

Shows the config every major MCP client needs to reach your `*.run.app/mcp` endpoint - Claude, OpenAI, Cursor, and VS Code - all pointing at the same URL. It builds the config objects and prints them; it does not send any request.

In [ ]:
# CONNECT REMOTE CLIENTS - one Cloud Run URL, many clients (they connect SERVER-SIDE).
MCP_URL = "https://netsetos-mcp-abc123-uc.a.run.app/mcp"

# 1) Claude via the Anthropic Messages API - the mcp_servers connector.
#    Send with client.beta.messages.create(**anthropic_request, betas=["mcp-client-2025-11-20"]).
anthropic_request = {
    "model": "claude-sonnet-4-6",
    "mcp_servers": [{"type": "url", "url": MCP_URL, "name": "netsetos",
                     "authorization_token": "..."}],   # optional OAuth bearer
    "tools": [{"type": "mcp_toolset", "mcp_server_name": "netsetos"}],   # required - names the server
    "messages": [{"role": "user", "content": "Add GST to 14999"}],
}
# 2) OpenAI via the Responses API - a tools entry of type "mcp":
openai_tools = [{"type": "mcp", "server_label": "netsetos", "server_url": MCP_URL,
                 "require_approval": "never"}]   # else every call blocks on manual approval
# 3) VS Code (.vscode/mcp.json) uses a top-level "servers" key:
vscode_mcp_json = {"servers": {"netsetos": {"type": "http", "url": MCP_URL}}}
# 3b) Cursor (~/.cursor/mcp.json) uses a top-level "mcpServers" key instead:
cursor_mcp_json = {"mcpServers": {"netsetos": {"type": "http", "url": MCP_URL}}}

print("One URL, every client:")
for name in ["Claude (mcp_servers)", "OpenAI (type=mcp)", "Cursor/VS Code (mcp.json)"]:
    print(f"  {name:26s} -> {MCP_URL}")

# Output:
#   One URL, every client:
#     Claude (mcp_servers)       -> https://netsetos-mcp-abc123-uc.a.run.app/mcp
#     OpenAI (type=mcp)          -> https://netsetos-mcp-abc123-uc.a.run.app/mcp
#     Cursor/VS Code (mcp.json)  -> https://netsetos-mcp-abc123-uc.a.run.app/mcp

**Explanation**

This is a reference cell: it assembles (but does not execute) the request/config shapes for four clients so you can see how each names and reaches the server. The payoff is that one deploy serves every client - the hosted connectors (Claude, OpenAI) connect server-side, while Cursor and VS Code read a small local `mcp.json`.

**How the code works - step by step**
- **`anthropic_request`** - Claude's `mcp_servers` connector (URL + optional `authorization_token`) paired with a `tools` entry of `type: "mcp_toolset"` that names the server; sent on the beta `mcp-client-2025-11-20` endpoint.
- **`openai_tools`** - OpenAI Responses API `tools` entry of `type: "mcp"` with `server_url` and `require_approval: "never"`.
- **`vscode_mcp_json` / `cursor_mcp_json`** - the two IDE configs; note the differing top-level key (`servers` vs `mcpServers`).
- **the `print` loop** - shows all clients resolving to the same `/mcp` URL.

**In one line:** four client configs, one shared HTTPS endpoint.

**What you'll see:** Prints `One URL, every client:` then Claude, OpenAI, and Cursor/VS Code each mapped to the same `https://netsetos-mcp-abc123-uc.a.run.app/mcp`. No live connection is made.

## 10 - The scale-to-zero cost shape

A cost model that contrasts scaling to zero against keeping one warm instance, so you can see the price of killing cold starts. The dollar figures are an illustrative model, not a real bill - the shape is the lesson.

In [ ]:
# SCALE-TO-ZERO vs a WARM min-instance - the cost SHAPE (illustrative model, not a bill).
SEC_PER_MONTH = 30 * 24 * 3600
ACTIVE = 0.000024          # model $/instance-second while serving a request
IDLE   = 0.0000025         # model $/instance-second while warm-but-idle (~10% of active)
def monthly_cost(busy_seconds, warm_instances):
    idle_seconds = warm_instances * SEC_PER_MONTH
    return round(busy_seconds * ACTIVE + idle_seconds * IDLE, 2)

busy = 10_000_000 * 0.4    # 10M requests x ~0.4s each = busy instance-seconds
scale0 = monthly_cost(busy, warm_instances=0)
warm1  = monthly_cost(busy, warm_instances=1)
print(f"  10M req/mo @ ~0.4s: busy compute ~= {int(busy):,} instance-seconds")
print(f"  scale-to-zero (min-instances=0): ~${scale0}/mo  (nothing billed while idle)")
print(f"  one warm instance (min-instances=1): ~${warm1}/mo  (+${round(warm1-scale0,2)} for zero cold starts)")
print("  numbers are an ILLUSTRATIVE model; the shape is the point: idle=0, warm=small fixed add-on.")

# Output:
#   10M req/mo @ ~0.4s: busy compute ~= 4,000,000 instance-seconds
#   scale-to-zero (min-instances=0): ~$96.0/mo  (nothing billed while idle)
#   one warm instance (min-instances=1): ~$102.48/mo  (+$6.48 for zero cold starts)
#   numbers are an ILLUSTRATIVE model; the shape is the point: idle=0, warm=small fixed add-on.

**Explanation**

`monthly_cost` charges an active rate for busy instance-seconds plus an idle rate for any warm instance held all month. Comparing `warm_instances=0` and `warm_instances=1` over the same busy compute isolates exactly what a warm floor costs: a small fixed add-on for zero cold starts, on top of identical serving cost.

**How the code works - step by step**
- **`ACTIVE` / `IDLE` rates** - model $/instance-second while serving vs while warm-but-idle (~10% of active).
- **`monthly_cost(busy_seconds, warm_instances)`** - busy seconds x active + warm-instance-seconds x idle.
- **`busy = 10M * 0.4`** - the serving compute for 10M requests at ~0.4s each.
- **`scale0` vs `warm1`** - the same load with zero vs one warm instance; the print shows the delta.

**In one line:** idle costs ~0; one warm instance adds a small fixed monthly charge for zero cold starts.

**What you'll see:** Prints `scale-to-zero ... ~$96.0/mo` versus `one warm instance ... ~$102.48/mo (+$6.48 for zero cold starts)`, with an explicit note that the numbers are illustrative.

## 11 - Canary traffic split with an error gate

Simulates a safe rollout: send a small slice of traffic to a new revision, measure its error rate, and gate on it - promote if healthy, roll back if not. It's a decision model that mirrors what the CI/CD pipeline does with real traffic weights.

In [ ]:
# CANARY traffic split + error gate - promote a good revision, roll back a bad one.
def canary(total, pct_v2, v2_error_rate, gate=0.05):
    v2 = total * pct_v2 // 100
    v1 = total - v2
    v2_errors = int(v2 * v2_error_rate)
    err = v2_errors / v2 if v2 else 0
    decision = "ROLL BACK to v1 (100%)" if err > gate else "promote v2 -> 100%"
    return v1, v2, v2_errors, round(err, 3), decision

for pct, rate in [(10, 0.01), (10, 0.20)]:
    v1, v2, e, err, dec = canary(1000, pct, rate)
    print(f"  split v1={100-pct}% v2={pct}%: v1={v1} v2={v2} reqs | v2 errors={e} (rate {err}) -> {dec}")
print("  deploy v2 with --no-traffic, send 10% canary, gate on error-rate, then promote or roll back.")

# Output:
#   split v1=90% v2=10%: v1=900 v2=100 reqs | v2 errors=1 (rate 0.01) -> promote v2 -> 100%
#   split v1=90% v2=10%: v1=900 v2=100 reqs | v2 errors=20 (rate 0.2) -> ROLL BACK to v1 (100%)
#   deploy v2 with --no-traffic, send 10% canary, gate on error-rate, then promote or roll back.

**Explanation**

`canary` splits total requests by percentage, computes the new revision's error rate over its slice, and compares it to a gate threshold to decide promote-or-rollback. Running it with a healthy v2 and a broken v2 shows both outcomes, illustrating why a small canary slice plus instant tagged-revision rollback is cheap insurance.

**How the code works - step by step**
- **`canary(total, pct_v2, v2_error_rate, gate=0.05)`** - splits traffic, counts v2 errors, and computes v2's error rate.
- **`decision`** - `ROLL BACK` if the rate exceeds the 5% gate, else `promote v2 -> 100%`.
- **the loop over `[(10, 0.01), (10, 0.20)]`** - a 1% error v2 (passes) and a 20% error v2 (fails).

**In one line:** send 10% canary, gate on error rate, then promote or roll back.

**What you'll see:** Prints the healthy split promoting v2 (rate 0.01) and the broken split rolling back (rate 0.2), then the summary line describing the `--no-traffic` -> canary -> gate flow.

## 12 - India-aware deployment

Shows the region-aware decisions for Indian users: deploy to Mumbai (`asia-south1`), wire in a real Indian government MCP server, and keep the compliance split straight. It prints a summary of the target; it makes no deploy or network call.

In [ ]:
# INDIA PRODUCTION - deploy to Mumbai; wire in a real Indian government MCP server.
# gcloud run deploy netsetos-mcp --source . --region asia-south1 --no-allow-unauthenticated
#   asia-south1 (Mumbai): ~5-20 ms in-country latency, all features, but NO free tier.

# MoSPI eSankhyiki is a real, official government MCP server (nso-india/esankhyiki-mcp, MIT):
mospi = {"type": "url", "url": "https://.../mospi-mcp", "name": "mospi-esankhyiki"}
#   started as a 7-dataset beta (Feb 2026), now ~two dozen official statistics datasets.

print("India deploy target:")
print("  region         = asia-south1 (Mumbai)   # no free tier; test on us-central1")
print("  indic wrappers = Bhashini (22 langs), Sarvam (Indic LLMs) as MCP servers")
print("  compliance     = RBI/SEBI residency (now) + DPDP full enforcement (2027-05-13)")

# Output:
#   India deploy target:
#     region         = asia-south1 (Mumbai)   # no free tier; test on us-central1
#     indic wrappers = Bhashini (22 langs), Sarvam (Indic LLMs) as MCP servers
#     compliance     = RBI/SEBI residency (now) + DPDP full enforcement (2027-05-13)

**Explanation**

This is a reference/summary cell - the `gcloud` line and the MoSPI server config are commented illustrations, and the body just prints the decisions. The three things to take away: Mumbai gives in-country latency but no free tier, real Indian MCP servers (MoSPI, Bhashini, Sarvam) exist today, and RBI/SEBI residency is a separate in-force rule from the DPDP Act's later penalties.

**How the code works - step by step**
- **commented `gcloud ... --region asia-south1`** - the same deploy as Step 5, just retargeted to Mumbai.
- **`mospi` dict** - a real government MCP server (`nso-india/esankhyiki-mcp`) wired in like any other remote server from Step 9.
- **the `print` block** - summarizes region (Mumbai, no free tier), Indic wrappers (Bhashini, Sarvam), and the compliance split.

**In one line:** deploy to Mumbai for latency/residency; keep RBI/SEBI (now) and DPDP (2027) compliance separate.

**What you'll see:** Prints the `India deploy target:` block - region `asia-south1`, the Indic wrappers, and `compliance = RBI/SEBI residency (now) + DPDP full enforcement (2027-05-13)`. No deploy runs.

entered below